# Food Leaf World Map: Offline Vector Version

This notebook is a comparison version of `leaf_world_map.ipynb`. It avoids online map tiles by rendering local Natural Earth country and Admin-1 vector polygons with `folium`.

The first setup cell downloads Natural Earth boundaries once into `data/boundaries/`. After that, the map can render without an online basemap. Country polygons provide the local background layer. Some provisional cuisine-region rows will not match official state/province boundaries; those are shown as fallback markers.

## Setup

Use the project Conda environment:

```bash
conda env update -f environment.yml --prune
conda activate foodmap
jupyter lab notebooks/leaf_world_map_offline_vector.ipynb
```

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

from leafmap import load_leaf_metadata, load_usage_data
from leafmap.offline_map import (
    attach_boundaries,
    boundary_match_summary,
    ensure_natural_earth_admin1,
    ensure_natural_earth_countries,
    load_admin1_boundaries,
    load_country_boundaries,
    render_offline_vector_map,
)

usage = load_usage_data(PROJECT_ROOT / "data" / "leaf_usage_regions.csv")
metadata = load_leaf_metadata(PROJECT_ROOT / "data" / "leaf_metadata.csv")

## Prepare Local Boundaries

This downloads Natural Earth country and Admin-1 data only if the shapefiles are missing locally. The map itself uses `tiles=None`, so it does not request online basemap tiles.

In [ ]:
boundary_dir = PROJECT_ROOT / "data" / "boundaries"
admin1_boundary_path = ensure_natural_earth_admin1(boundary_dir)
country_boundary_path = ensure_natural_earth_countries(boundary_dir)
boundaries = load_admin1_boundaries(admin1_boundary_path)
country_boundaries = load_country_boundaries(country_boundary_path)
admin1_boundary_path, country_boundary_path

## Boundary Match Coverage

Rows that do not match official Natural Earth state/province names are still displayed as fallback markers. Later, we can improve this by adding a manual alias table or converting city/cuisine-region rows into official admin-1 rows.

In [ ]:
usage_with_boundaries = attach_boundaries(usage, boundaries)
boundary_match_summary(usage_with_boundaries)

In [ ]:
unmatched = usage_with_boundaries[usage_with_boundaries.geometry.isna()]
unmatched[["leaf_name", "country", "admin1", "region_label"]].sort_values(["country", "admin1", "leaf_name"])

## Interactive Offline Vector Map

Selected official state/province matches render as polygons. Unmatched rows render as local fallback markers. Orange polygons and black outline markers indicate selected-leaf overlap.

In [ ]:
leaf_options = [(row.leaf_name, row.leaf_id) for row in metadata.sort_values("leaf_name").itertuples()]
selector = widgets.SelectMultiple(
    options=leaf_options,
    value=tuple(value for _, value in leaf_options[:3]),
    description="Leaves",
    rows=min(12, len(leaf_options)),
    layout=widgets.Layout(width="260px"),
)
output = widgets.Output(layout=widgets.Layout(width="100%"))

def update_offline_map(change=None):
    with output:
        clear_output(wait=True)
        display(render_offline_vector_map(usage, metadata, list(selector.value), boundaries, country_boundaries))

selector.observe(update_offline_map, names="value")
update_offline_map()
widgets.HBox([selector, output])

## Notes For Comparing Versions

The original notebook uses online background tiles and point markers. This notebook uses local vector polygons and no online tiles. Polygon quality depends on matching the provisional `country + admin1` values to Natural Earth's official names.